In [6]:
# 🌍 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 📦 2. Install needed libraries
!pip install -q faiss-cpu sentence-transformers transformers accelerate

# 📚 3. Import all libraries
import pandas as pd
import numpy as np
import faiss
from scipy.spatial import cKDTree
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 📂 4. Load Soil Data
soil_df = pd.read_csv('/content/drive/MyDrive/FarmWise/soil_data_cleaned.csv')  # Adjust path if needed

# 🗺️ 5. Build KDTree to find nearest soil sample by coordinates
soil_coords = soil_df[['latitude', 'longitude']].values
tree = cKDTree(soil_coords)

def get_nearest_soil(lat, lon):
    distance, idx = tree.query([lat, lon])
    return soil_df.iloc[idx]

# 📖 6. Define the Extended Soil Knowledge Base
knowledge_base = [
    "pH < 5.5: Strongly acidic soil, aluminum toxicity risk. Lime recommended.",
    "5.5 <= pH <= 6.5: Slightly acidic, optimal for most crops.",
    "6.5 < pH <= 7.5: Neutral pH, ideal nutrient availability.",
    "pH > 7.5: Alkaline soil. Risk of nutrient lock-out. Sulfur treatment advised.",
    "Organic Carbon < 1%: Low fertility, compost or manure needed.",
    "Organic Carbon 1%-2%: Moderate fertility, maintain organic matter.",
    "Organic Carbon > 2%: High fertility, excellent soil health.",
    "Clay > 35%: Heavy soil, drainage issues likely. Raised beds recommended.",
    "15% < Clay <= 35%: Loamy soil, good texture for crops.",
    "Clay < 15%: Sandy soil, high drainage, low nutrient retention.",
    "Nitrogen < 0.1%: Deficiency, recommend legumes or N fertilizer.",
    "Bulk Density > 1.6 g/cm³: Compaction risk. Reduce tillage pressure.",
    "CEC < 10 cmol/kg: Poor nutrient retention, add organic matter.",
    "CEC > 25 cmol/kg: Good fertility potential.",
    "Low water retention: Drought risk. Mulch and compost advised.",
    "High water retention: Monitor for waterlogging, especially in heavy soils."
]

# 🧱 7. Embed the Knowledge Base using Sentence Transformers + FAISS
embedder = SentenceTransformer('all-MiniLM-L6-v2')
knowledge_embeddings = embedder.encode(knowledge_base)

dimension = knowledge_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(knowledge_embeddings))

def retrieve_knowledge(query, top_k=5):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    return [knowledge_base[i] for i in indices[0]]

# 🧠 8. Load Free Phi-2 Model
model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

soil_expert = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    temperature=0.4,
    repetition_penalty=1.1
)

# 📝 9. Function to Generate Soil Report
def generate_soil_report(lat, lon):
    nearest_soil = get_nearest_soil(lat, lon)
    soil_features = nearest_soil.to_dict()

    query_string = " ".join([f"{k}={v}" for k, v in soil_features.items()])
    top_knowledge = retrieve_knowledge(query_string)

    prompt = f"""
You are a professional soil scientist.
Given the following soil data:

{soil_features}

And the following agronomic knowledge:

{top_knowledge}

Write a full detailed soil analysis report, explaining:
- The soil health condition
- Risks and issues
- Clear management and improvement recommendations
- Suitable crop types
"""

    # Call the local free model
    result = soil_expert(prompt)[0]["generated_text"]

    return result

# 📍 10. Test on a Coordinate
latitude = 36.8
longitude = 10.1

report = generate_soil_report(latitude, longitude)
print(report)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.4` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are a professional soil scientist.
Given the following soil data:

{'latitude': 10.998805256869773, 'longitude': 30.001129943502825, 'bulk density': 1.61, 'cation exchange capacity (at ph 7)': 15.0, 'clay content': 38.3, 'coarse fragments': 90.0, 'nitrogen': 100.0, 'silt': 24.7, 'soil organic carbon': 0.63, 'vol. water content at -10 kpa': 386.0, 'vol. water content at -1500 kpa': 162.0, 'vol. water content at -33 kpa': 305.0, 'organic carbon density': 118.0, 'ph water': 6.6, 'soil classification': 'Petrogypsic Gypsisols (PGY)'}

And the following agronomic knowledge:

['Clay < 15%: Sandy soil, high drainage, low nutrient retention.', '15% < Clay <= 35%: Loamy soil, good texture for crops.', 'Clay > 35%: Heavy soil, drainage issues likely. Raised beds recommended.', 'Organic Carbon > 2%: High fertility, excellent soil health.', 'pH > 7.5: Alkaline soil. Risk of nutrient lock-out. Sulfur treatment advised.']

Write a full detailed soil analysis report, explaining:
- The soil health